# Smart Household Finance Agent - Demo Notebook

This notebook demonstrates the **agentic capabilities** of the Finance Chat Agent.

## Agent Architecture
```
User Question
     |
     v
Step 1: Intent Classification (Gemini LLM)
     |
     v
Step 2: Data Retrieval (Supabase - Tool Usage)
     |
     v
Step 3: Context Building (with Conversation Memory)
     |
     v
Step 4: Response Generation (Gemini LLM)
     |
     v
Step 5: Memory Update
```

### Agentic Capabilities Demonstrated
1. **Goal-oriented** - Answers financial questions with real data
2. **Multi-step reasoning** - Classify, retrieve, analyze, respond
3. **Tool usage** - Queries Supabase database
4. **Memory** - Retains conversation history
5. **Decision-making** - Selects data retrieval strategy based on intent
6. **Autonomy** - Completes full pipeline without human intervention

## Setup

In [ ]:
import sys
sys.path.insert(0, '.')

from expense_manager import ExpenseManager
from chat_agent import FinanceChatAgent
from vision_agent import VisionAgent
from config import DEFAULT_BUDGET
import json

# Initialize
em = ExpenseManager()
settings = em.get_user_settings()
budget = float(settings.get('monthly_budget', DEFAULT_BUDGET)) if settings else DEFAULT_BUDGET
nickname = settings.get('nickname', 'User') if settings else 'User'

print(f"User: {nickname}")
print(f"Monthly Budget: P {budget:,.2f}")
print(f"Database connected successfully.")

In [ ]:
# Initialize the Chat Agent
agent = FinanceChatAgent(em, budget)
print("Finance Chat Agent initialized.")
print(f"Available intents: {json.dumps(agent._classify_intent.__code__.co_consts, default=str)[:200]}")

## Agent Demo: Step-by-Step Execution

Let's trace through the agent's full pipeline for a single query.

In [ ]:
# STEP-BY-STEP AGENT DEMO
test_question = "Am I over budget this month?"
print(f"USER: {test_question}")
print("=" * 60)

# Step 1: Intent Classification
intent = agent._classify_intent(test_question)
print(f"\nStep 1 - Intent Classification: {intent}")

# Step 2: Data Retrieval
context = agent._retrieve_data(intent, test_question)
print(f"\nStep 2 - Data Retrieved:")
print(json.dumps(context, indent=2, default=str)[:500])

# Step 3: Full agent response
result = agent.chat(test_question)
print(f"\nStep 3-5 - Agent Response:")
print(result['response'])
print(f"\nAgent Steps:")
for s in result['steps']:
    print(f"  {s}")

## Test Scenarios (5-10 Test Cases)

Below we run multiple test scenarios and document the results.

In [ ]:
# Define test cases
test_cases = [
    {
        "id": 1,
        "question": "Am I over budget this month?",
        "expected_intent": "budget_check",
        "expected_behavior": "Should report current spending vs budget with specific numbers",
    },
    {
        "id": 2,
        "question": "What are my top 5 biggest expenses?",
        "expected_intent": "top_expenses",
        "expected_behavior": "Should list top expenses sorted by amount with dates",
    },
    {
        "id": 3,
        "question": "How much did I spend on groceries?",
        "expected_intent": "category_analysis",
        "expected_behavior": "Should show food/grocery category total with breakdown",
    },
    {
        "id": 4,
        "question": "Compare this month vs last month spending",
        "expected_intent": "monthly_comparison",
        "expected_behavior": "Should compare totals and categories between two months",
    },
    {
        "id": 5,
        "question": "How can I save money?",
        "expected_intent": "savings_tips",
        "expected_behavior": "Should analyze spending patterns and suggest specific cuts",
    },
    {
        "id": 6,
        "question": "What payment method do I use the most?",
        "expected_intent": "payment_analysis",
        "expected_behavior": "Should show payment method breakdown with amounts",
    },
    {
        "id": 7,
        "question": "How much does rice cost now?",
        "expected_intent": "item_price",
        "expected_behavior": "Should show price history for rice if available",
    },
    {
        "id": 8,
        "question": "Give me a summary of my finances",
        "expected_intent": "general_query",
        "expected_behavior": "Should provide overall spending summary with budget status",
    },
    {
        "id": 9,
        "question": "What did I spend the most on in utilities?",
        "expected_intent": "category_analysis",
        "expected_behavior": "Should focus on utilities category with specific items",
    },
    {
        "id": 10,
        "question": "Based on our conversation, what should I prioritize?",
        "expected_intent": "general_query",
        "expected_behavior": "Should reference previous conversation context (memory test)",
    },
]

In [ ]:
# Run all test cases
results = []

for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"TEST CASE {tc['id']}: {tc['question']}")
    print(f"Expected Intent: {tc['expected_intent']}")
    print(f"Expected Behavior: {tc['expected_behavior']}")
    print(f"{'='*60}")
    
    try:
        result = agent.chat(tc['question'])
        intent_match = result['intent'] == tc['expected_intent']
        
        print(f"\nActual Intent: {result['intent']} {'[MATCH]' if intent_match else '[MISMATCH]'}")
        print(f"Response Preview: {result['response'][:300]}...")
        print(f"Steps: {result['steps']}")
        
        results.append({
            "id": tc['id'],
            "question": tc['question'],
            "expected_intent": tc['expected_intent'],
            "actual_intent": result['intent'],
            "intent_match": intent_match,
            "response_length": len(result['response']),
            "has_data": result['data_retrieved'],
            "status": "PASS" if intent_match and result['data_retrieved'] else "PARTIAL",
            "response_preview": result['response'][:200],
        })
    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "id": tc['id'],
            "question": tc['question'],
            "status": "FAIL",
            "error": str(e),
        })

In [ ]:
# Results Summary Table
import pandas as pd

df = pd.DataFrame(results)
display_cols = ['id', 'question', 'expected_intent', 'actual_intent', 'intent_match', 'status']
available_cols = [c for c in display_cols if c in df.columns]
print("\nTEST RESULTS SUMMARY")
print("=" * 80)
print(df[available_cols].to_string(index=False))

passed = sum(1 for r in results if r.get('status') == 'PASS')
total = len(results)
print(f"\nResult: {passed}/{total} tests passed")
print(f"Memory test: Agent has {len(agent.conversation_history)} messages in history")

## Memory Demonstration

The agent retains conversation context across multiple exchanges.

In [ ]:
# Show conversation memory
print(f"Conversation history length: {len(agent.conversation_history)} messages")
print(f"({len(agent.conversation_history)//2} exchanges)\n")

for i, msg in enumerate(agent.conversation_history[-6:]):
    role = 'USER' if msg['role'] == 'user' else 'AGENT'
    print(f"[{role}]: {msg['content'][:150]}...\n")

## Vision Agent Demo (Receipt Scanning)

The Vision Agent demonstrates tool usage by calling Gemini Vision API to extract structured data from receipt images.

In [ ]:
# Vision Agent - Architecture demo
va = VisionAgent()
print("Vision Agent initialized.")
print("\nVision Agent Pipeline:")
print("  1. User uploads receipt image")
print("  2. Image sent to Gemini Vision API (tool usage)")
print("  3. LLM extracts: store name, date, items, brands, categories")
print("  4. JSON response parsed and validated")
print("  5. Items presented for user review (human-in-the-loop)")
print("  6. Confirmed items saved to Supabase (automated action)")
print("\nThis demonstrates:")
print("  - Multi-step reasoning (extract -> categorize -> save)")
print("  - Tool usage (Gemini API + Supabase)")
print("  - Decision-making (auto-categorization per item)")
print("  - Autonomy (batch processing without per-item intervention)")

## Limitations and Failure Cases

| Limitation | Description |
|---|---|
| Date misreading | Gemini may misread receipt dates (e.g., reading 2014 instead of 2024). Mitigated by defaulting to today. |
| Blurry receipts | Low-quality images produce incomplete or incorrect extractions. |
| Non-receipt images | Uploading a non-receipt image may produce nonsensical data. |
| Intent misclassification | Ambiguous questions may be classified under the wrong intent. |
| No offline mode | Requires internet for both Gemini API and Supabase. |
| Single user | Currently supports only one user profile per deployment. |
| Memory scope | Conversation memory is session-based; it resets when the app restarts. |

## Responsible AI Reflection

### Privacy and Data Security
This agent processes financial data, which is inherently sensitive. Receipt images are sent to Google's Gemini API for processing, meaning users' purchase history temporarily passes through a third-party service. While Google's API terms provide data handling assurances, users should be aware that their spending data leaves the local environment during receipt scanning. All persistent storage uses Supabase with Row Level Security policies, and API keys are managed through environment variables rather than hardcoded values.

### Accuracy and Reliability
The AI agent is not infallible. During development, we encountered cases where Gemini misread receipt dates (interpreting 2024 as 2014) and occasionally miscategorized items. The system mitigates this by making all AI-extracted data fully editable before saving, defaulting receipt dates to today rather than trusting the OCR output, and allowing users to review every line item. However, users must understand that AI-generated categorizations and insights are suggestions, not financial advice. The agent explicitly presents its reasoning steps so users can evaluate the basis of its responses.

### Bias Considerations
The expense categories are predefined and may not reflect all cultural spending patterns. The category list was designed for a Philippine household context (including GCash and Maya as payment methods), but users from different regions may find the categories insufficient. The savings tips generated by the LLM may carry implicit biases about what constitutes "essential" vs "non-essential" spending, which varies by household.

### Transparency
The AI Assistant page displays the agent's reasoning steps for every response: the classified intent, the data retrieval strategy used, and the amount of context provided to the LLM. This transparency allows users to understand why the agent gave a particular answer and to identify cases where it may have retrieved the wrong data. The system never saves data silently; every expense entry requires explicit user confirmation.

### Safe Interaction Guidelines
Users should always verify AI-extracted amounts before saving, treat spending recommendations as suggestions rather than professional financial advice, and understand that the system's memory is session-based. The agent is designed as a tracking and analysis tool, not a replacement for professional financial planning.